# LAV-DF private, authenticated baseline workflow

**Run setup and diagnostics in order. Do not train until both the environment and dataset audits pass.** Review provider terms and confirm that Colab and your private storage are permitted. Never put tokens, videos, features, identity metadata, reports or weights in public Git.

The Colab control kernel may show Python 3.13: this is expected. Model loading, extraction, dataset processing, training and evaluation run in a separate Python 3.12.10 virtual environment at `/content/lavdf-cuda-env/bin/python`. Its Python installation is under `/content/lavdf-python312-bootstrap`. The next diagnostic prints both interpreters and runs a CUDA calculation in the project environment. Select a T4 GPU and use a fresh Linux x86-64 Colab session with sufficient free storage and internet access for the setup downloads.

If setup is interrupted, rerun the setup cell: verified files and the virtual environment are reused, and package installation resumes. If it reports an unrecognized existing repository or environment, preserve any needed files and start a fresh Colab runtime; the setup intentionally does not delete unknown files. If a download or installation fails, check the error and connection, then rerun the cell. Private checkpoints and Hugging Face authentication are only needed in later cells.


In [1]:
GPU_DIAGNOSTIC_PASSED = MODEL_SMOKE_PASSED = DATASET_VERIFIED = False
import hashlib, os, re, subprocess, sys, tarfile, tempfile, urllib.request
from pathlib import Path

# Colab can set Python-specific paths for its control kernel; do not pass them to Python 3.12.
PROJECT_ENV = {k: v for k, v in os.environ.items() if k not in ('PYTHONPATH', 'PYTHONHOME')}

REPO_COMMIT = '4979a43a0cb016972c7ae295365e799a34d0e520'  # paste the exact reviewed 40-character commit from the instructions
REPO = Path('/content/deepfake-detection-dtw')
BOOTSTRAP = Path('/content/lavdf-python312-bootstrap')
VENV = Path('/content/lavdf-cuda-env')
PYTHON = str(VENV / 'bin/python')
PYTHON_SHA256 = '3b7de567f4a8a1c3e5be2d51af3023dd3c1075cfacb6cd8efb511e0ef4b31fa8'
PYTHON_URL = 'https://github.com/astral-sh/python-build-standalone/releases/download/20250529/cpython-3.12.10%2B20250529-x86_64-unknown-linux-gnu-install_only.tar.gz'
assert re.fullmatch(r'[0-9a-f]{40}', REPO_COMMIT), 'Set REPO_COMMIT to the reviewed commit SHA'
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Pradeepbeleri/deepfake-detection-dtw.git', str(REPO)], check=True)
assert not subprocess.check_output(['git','-C',str(REPO),'status','--porcelain','--untracked-files=all'],text=True).strip(), 'Repository has local changes; preserve them and use a clean checkout'
subprocess.run(['git','-C',str(REPO),'fetch','origin',REPO_COMMIT],check=True)
subprocess.run(['git','-C',str(REPO),'switch','--detach',REPO_COMMIT],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip() == REPO_COMMIT

# Refuse to reuse an unrelated or partially created installation. Never overwrite it.
marker = BOOTSTRAP / '.verified-sha256'
if not BOOTSTRAP.exists():
    with tempfile.TemporaryDirectory(prefix='lavdf-python-', dir=BOOTSTRAP.parent) as temp:
        archive = Path(temp) / 'python.tar.gz'
        urllib.request.urlretrieve(PYTHON_URL, archive)
        with archive.open('rb') as stream:
            actual = hashlib.file_digest(stream, 'sha256').hexdigest()
        if actual != PYTHON_SHA256:
            raise RuntimeError('Python archive SHA-256 mismatch; no archive was used')
        # Extract and mark in a temporary sibling: interruption cannot leave a
        # partially extracted installation at the final path.
        staged = Path(temp) / 'bootstrap'
        staged.mkdir(mode=0o700)
        with tarfile.open(archive, 'r:gz') as tar:
            tar.extractall(path=staged, filter='data')
        (staged / '.verified-sha256').write_text(PYTHON_SHA256)
        staged.rename(BOOTSTRAP)
assert marker.is_file() and marker.read_text().strip() == PYTHON_SHA256, 'Existing bootstrap is incomplete or unrecognized. Preserve your files and start a fresh Colab runtime.'
standalone = BOOTSTRAP / 'python/bin/python3.12'
assert subprocess.check_output([str(standalone),'-c','import sys; print(".".join(map(str,sys.version_info[:3])))'],text=True,env=PROJECT_ENV).strip() == '3.12.10'
venv_marker = VENV / '.lavdf-python-sha256'
if not VENV.exists():
    # Mark our directory before provisioning so an interrupted venv can be resumed.
    VENV.mkdir(mode=0o700)
    venv_marker.write_text(PYTHON_SHA256)
assert venv_marker.is_file() and venv_marker.read_text().strip() == PYTHON_SHA256, 'Existing virtual environment is unrecognized. Preserve your files and start a fresh Colab runtime.'
venv_complete = VENV / '.venv-complete'
if not venv_complete.is_file() or not Path(PYTHON).is_file():
    subprocess.run([str(standalone),'-m','venv',str(VENV)],env=PROJECT_ENV,check=True)
    venv_complete.write_text('ok')
print('Project interpreter:', PYTHON, flush=True)

# GPU wheels and CUDA libraries are confined to this virtual environment.
cuda_packages = ['nvidia-cuda-nvrtc-cu12','nvidia-cuda-runtime-cu12','nvidia-cuda-cupti-cu12','nvidia-cudnn-cu12','nvidia-cublas-cu12','nvidia-cufft-cu12','nvidia-curand-cu12','nvidia-cusolver-cu12','nvidia-cusparse-cu12','nvidia-nccl-cu12','nvidia-nvtx-cu12','nvidia-nvjitlink-cu12','triton']
subprocess.run([PYTHON,'-m','pip','install','--index-url','https://pypi.org/simple','-c',str(REPO/'requirements-colab-cu124.txt'),*cuda_packages],env=PROJECT_ENV,check=True)
subprocess.run([PYTHON,'-m','pip','install','--no-deps','--index-url','https://download.pytorch.org/whl/cu124','torch==2.5.1+cu124','torchvision==0.20.1+cu124'],env=PROJECT_ENV,check=True)
subprocess.run([PYTHON,'-m','pip','install','-c',str(REPO/'requirements-colab-cu124.txt'),'-e',str(REPO),'huggingface_hub==0.30.2'],env=PROJECT_ENV,check=True)
subprocess.run([PYTHON,'-m','pip','check'],env=PROJECT_ENV,check=True)
print('Setup completed. Run the next cell to check both interpreters and CUDA.', flush=True)


Project interpreter: /content/lavdf-cuda-env/bin/python
Setup completed. Run the next cell to check both interpreters and CUDA.


In [3]:
GGPU_DIAGNOSTIC_PASSED = False
import os, platform, subprocess, sys
from pathlib import Path

print('=== Colab control kernel (not used for project ML) ===', flush=True)
print('Python version:', platform.python_version(), flush=True)
print('Python executable:', sys.executable, flush=True)
print('Python 3.13 here is OK; the project uses the separate interpreter below.', flush=True)

PYTHON = '/content/lavdf-cuda-env/bin/python'
if not Path(PYTHON).is_file():
    raise RuntimeError('Project interpreter not found. Run the setup cell above and resolve any error before retrying.')

DIAGNOSTIC = r"""
import importlib.metadata as md, platform, sys
import torch, torchvision, whisper, mediapipe
import deepfake_dtw.encoders
print('\n=== Project ML environment (used by project commands) ===', flush=True)
print('Python version:', platform.python_version(), flush=True)
print('Python executable:', sys.executable, flush=True)
print('PyTorch:', torch.__version__, flush=True)
print('torchvision:', torchvision.__version__, flush=True)
print('PyTorch CUDA build:', torch.version.cuda, flush=True)
print('Whisper:', md.version('openai-whisper'), '| MediaPipe:', mediapipe.__version__, flush=True)
available = torch.cuda.is_available()
print('CUDA available:', available, flush=True)
print('GPU name:', torch.cuda.get_device_name(0) if available else 'Unavailable', flush=True)
if sys.version_info[:3] != (3, 12, 10):
    raise RuntimeError('Unexpected project Python version; rerun setup or use a fresh runtime.')
if torch.__version__ != '2.5.1+cu124' or torchvision.__version__ != '0.20.1+cu124' or torch.version.cuda != '12.4':
    raise RuntimeError('Unexpected PyTorch CUDA build; rerun setup and check pip errors.')
if not available:
    raise RuntimeError('CUDA unavailable in the project interpreter. Select a T4 GPU in Runtime > Change runtime type, reconnect, rerun setup and check the runtime driver.')
x = torch.tensor([1.0, 2.0], device='cuda')
if (x*x).sum().item() != 5.0:
    raise RuntimeError('CUDA tensor calculation returned an unexpected result.')
torch.cuda.synchronize()
print('CUDA tensor operation: PASSED', flush=True)
"""

# Configure diagnostic environment. Explicitly override the MPLBACKEND to a non-interactive backend 'Agg'
# to prevent matplotlib from attempting to load Colab's interactive inline backend.
diag_env = {k: v for k, v in os.environ.items() if k not in ('PYTHONPATH', 'PYTHONHOME')}
diag_env['MPLBACKEND'] = 'Agg'

try:
    result = subprocess.run(
        [PYTHON, '-c', DIAGNOSTIC],
        env=diag_env,
        capture_output=True,
        text=True,
        check=True
    )
    print(result.stdout)
except subprocess.CalledProcessError as exc:
    print("--- DIAGNOSTIC STDOUT ---")
    print(exc.stdout)
    print("--- DIAGNOSTIC STDERR ---")
    print(exc.stderr)
    raise RuntimeError('Project diagnostic failed; see the captured stdout and stderr above. Resolve it and rerun this cell before continuing.') from exc
GPU_DIAGNOSTIC_PASSED = True
print('Environment check passed. Continue to private checkpoint configuration.', flush=True)


=== Colab control kernel (not used for project ML) ===
Python version: 3.13.16
Python executable: /usr/bin/python3
Python 3.13 here is OK; the project uses the separate interpreter below.

=== Project ML environment (used by project commands) ===
Python version: 3.12.10
Python executable: /content/lavdf-cuda-env/bin/python
PyTorch: 2.5.1+cu124
torchvision: 0.20.1+cu124
PyTorch CUDA build: 12.4
Whisper: 20250625 | MediaPipe: 0.10.21
CUDA available: True
GPU name: Tesla T4
CUDA tensor operation: PASSED

Environment check passed. Continue to private checkpoint configuration.


In [4]:
import os, json, subprocess, sys
from pathlib import Path

PROJECT_ENV = {k: v for k, v in os.environ.items() if k not in ('PYTHONPATH', 'PYTHONHOME')}

REPO = Path('/content/deepfake-detection-dtw')
PYTHON = '/content/lavdf-cuda-env/bin/python'
# Mount only the Colab control kernel's Drive integration; ML uses the isolated interpreter.
if not Path('/content/drive/MyDrive').is_dir():
    from google.colab import drive
    drive.mount('/content/drive')
PRIVATE = Path('/content/drive/MyDrive/deepfake-private')  # only if provider/institution authorize this storage
MEDIA = PRIVATE / 'lavdf' / 'media'
FEATURES = PRIVATE / 'lavdf' / 'features'
REPORTS = PRIVATE / 'lavdf' / 'reports'
PROVENANCE = PRIVATE / 'lavdf' / 'curated-provenance.json'
PLAN = PRIVATE / 'lavdf' / 'plan-source-groups.json'  # change this path for a separate experiment
METADATA_LOCAL = PRIVATE / 'lavdf' / 'metadata.min.json'
WHISPER = PRIVATE / 'tiny.en.pt'
DINO = PRIVATE / 'dinov2_vits14_pretrain.pth'
DINO_SOURCE = PRIVATE / 'dinov2-src'  # clean pinned official checkout
DINO_SHA256 = 'b938bf1bc15cd2ec0feacfe3a1bb553fe8ea9ca46a7e1d8d00217f29aef60cd9'  # user-supplied transfer hash; verify your file provenance
assert (REPO / 'src/deepfake_dtw/lavdf.py').exists() and Path(PYTHON).is_file(), 'Complete repository environment setup first'
assert PRIVATE.is_dir(), 'Mount and create your authorized private storage root before continuing'
assert len(DINO_SHA256) == 64 and WHISPER.is_file() and DINO.is_file(), 'Provide and verify both permitted model files'
for directory in (MEDIA, FEATURES, REPORTS):
    directory.mkdir(parents=True, mode=0o700, exist_ok=True)


Mounted at /content/drive


In [17]:
print("Model smoke test passed:", MODEL_SMOKE_PASSED)

Model smoke test passed: True


In [16]:
MODEL_SMOKE_PASSED = False
assert globals().get('GPU_DIAGNOSTIC_PASSED', False), 'Run the GPU diagnostic successfully first'
# Required model-loading check. It performs no classifier training.
MODEL_SMOKE = "import json, numpy as np, sys, torch\nfrom deepfake_dtw.encoders import WhisperAudioEncoder, DinoVisualEncoder\nfrom deepfake_dtw.preprocessing import prepare_audio, prepare_video\ndevice=torch.device('cuda')\na=WhisperAudioEncoder(device=device,mode='pretrained',checkpoint=sys.argv[1])\nv=DinoVisualEncoder(device=device,checkpoint=sys.argv[2],sha256=sys.argv[3],source_dir=sys.argv[4])\naudio=prepare_audio(np.zeros(16000,dtype=np.float32),16000)\nvideo=prepare_video(np.zeros((1,64,64,3),dtype=np.uint8),[0.0])\naf=a.extract(audio); vf=v.extract(video)\nassert torch.isfinite(af.values).all() and torch.isfinite(vf.values).all()\nassert af.values.shape==(1,1500,384) and vf.values.shape==(1,384)\ntorch.cuda.synchronize()\nprint(json.dumps({'model_interpreter':sys.executable,'device':str(device),'gpu':torch.cuda.get_device_name(0),'audio_shape':list(af.values.shape),'video_shape':list(vf.values.shape),'audio_parameters':sum(p.numel() for p in a.model.parameters()),'visual_parameters':sum(p.numel() for p in v.model.parameters()),'gpu_peak_bytes':torch.cuda.max_memory_allocated()},indent=2))\n"
model_env = dict(PROJECT_ENV, XFORMERS_DISABLED='1', OMP_NUM_THREADS='2', CUBLAS_WORKSPACE_CONFIG=':4096:8')
subprocess.run([PYTHON,'-c',MODEL_SMOKE,str(WHISPER),str(DINO),DINO_SHA256,str(DINO_SOURCE)],env=model_env,check=True)
MODEL_SMOKE_PASSED = True


In [15]:
DINO_SOURCE = Path('/content/dinov2-src')
print("DINOv2 source:", DINO_SOURCE)

DINOv2 source: /content/dinov2-src


In [11]:
import subprocess

result = subprocess.run(
    [
        PYTHON,
        "-c",
        MODEL_SMOKE,
        str(WHISPER),
        str(DINO),
        DINO_SHA256,
        str(DINO_SOURCE),
    ],
    env=model_env,
    capture_output=True,
    text=True,
)

print("EXIT CODE:", result.returncode)
print("\n=== STANDARD OUTPUT ===")
print(result.stdout[-4000:])
print("\n=== ACTUAL ERROR ===")
print(result.stderr[-6000:])

EXIT CODE: 1

=== STANDARD OUTPUT ===


=== ACTUAL ERROR ===
Traceback (most recent call last):
  File "/content/deepfake-detection-dtw/src/deepfake_dtw/encoders.py", line 117, in verify_dinov2_source
    status = subprocess.run(['git', '-C', str(root), 'status', '--porcelain', '--untracked-files=all'], check=True, capture_output=True, text=True, timeout=10).stdout
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/lavdf-python312-bootstrap/python/lib/python3.12/subprocess.py", line 550, in run
    stdout, stderr = process.communicate(input, timeout=timeout)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/lavdf-python312-bootstrap/python/lib/python3.12/subprocess.py", line 1209, in communicate
    stdout, stderr = self._communicate(input, endtime, timeout)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/

In [13]:
from pathlib import Path
import subprocess

drive_source = Path('/content/drive/MyDrive/deepfake-private/dinov2-src')
local_source = Path('/content/dinov2-src')

assert drive_source.is_dir(), "DINOv2 source missing from Drive"

if not local_source.exists():
    subprocess.run(
        ['cp', '-a', str(drive_source), str(local_source)],
        check=True
    )

expected_commit = '7764ea0f912e53c92e82eb78a2a1631e92725fc8'

actual_commit = subprocess.check_output(
    ['git', '-C', str(local_source), 'rev-parse', 'HEAD'],
    text=True
).strip()

assert actual_commit == expected_commit, "Unexpected DINOv2 commit"

subprocess.run(
    ['git', '-C', str(local_source), 'status', '--porcelain', '--untracked-files=all'],
    check=True,
    timeout=30
)

DINO_SOURCE = local_source

print("Local DINOv2 source verified:", actual_commit)

Local DINOv2 source verified: 7764ea0f912e53c92e82eb78a2a1631e92725fc8


## Authorized archive and private storage

The provider supplies `LAV-DF.tar`, not individual media files. Keep its 25.61 GB archive and Hugging Face cache in authorized private Drive storage so they survive runtime resets. Check Drive capacity for the archive, selected videos, features and temporary download files. Hugging Face resumes incomplete downloads where supported; a reset can still require retrying. Authentication is interactive and the token is never placed in code or Git. Use only after reviewing provider and underlying dataset terms.


In [27]:
# Only authenticate when the archive is absent. The download and cache stay private.
from getpass import getpass
ARCHIVE_DIR = PRIVATE / 'lavdf' / 'archive'
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True, mode=0o700)
ARCHIVE = ARCHIVE_DIR / 'LAV-DF.tar'
ARCHIVE_SHA256 = '2abc0646cf1469f0bebec60a792fec8b6d670cd9e6133105139110c4316b20a4'
HF_HOME = PRIVATE / 'lavdf' / 'hf-private'
HF_HOME.mkdir(parents=True, mode=0o700, exist_ok=True)
os.chmod(HF_HOME, 0o700)
hf_env = dict(PROJECT_ENV, HF_HOME=str(HF_HOME))
if not ARCHIVE.is_file():
    token = getpass('Hugging Face read token (not echoed): ')
    try:
        subprocess.run([PYTHON, '-c', "import sys; from huggingface_hub import login; login(token=sys.stdin.read().strip(), add_to_git_credential=False)"], input=token+'\n', text=True, env=hf_env, check=True)
    finally:
        del token
    download_script = "import sys; from huggingface_hub import hf_hub_download; print(hf_hub_download(repo_id='ControlNet/LAV-DF', repo_type='dataset', filename='LAV-DF.tar', local_dir=sys.argv[1]))"
    subprocess.run([PYTHON, '-c', download_script, str(ARCHIVE_DIR)], env=hf_env, check=True)
assert ARCHIVE.is_file(), 'Archive not found; check authorized provider access and retry.'
print('Private archive available. Its complete SHA-256 is verified before any extraction.')


Hugging Face read token (not echoed): ··········
Private archive available. Its complete SHA-256 is verified before any extraction.


## Plan an explicitly scoped source-group baseline

The default `source-group-only` mode uses provider original/source links and official partitions. It makes **no identity-disjointness claim**: different recordings of one participant or undisclosed audio sources may cross partitions. The importer excludes untraceable or cross-partition source links rather than inventing identities, and stops if it cannot construct the requested partitions. The provider and upstream dataset conditions continue to apply. The existing `identity` mode remains available if you have a complete audited sidecar; set `PLAN_MODE` accordingly and use a separate plan path.


In [20]:
# Select scope explicitly; the default does not require an identity sidecar.
import hashlib
from collections import Counter
PLAN_MODE = 'source-group-only'  # or 'identity' if a complete audited identity sidecar is available
USAGE_RIGHTS = 'User reports authorized noncommercial academic research; provider and VoxCeleb2 conditions apply.'
assert PLAN_MODE in ('source-group-only', 'identity')
assert METADATA_LOCAL.is_file(), f'Provide the provider metadata at {METADATA_LOCAL}'
PLAN_STAMP = PLAN.with_suffix('.inputs.json')
def digest_file(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()
inputs = {'metadata_sha256': digest_file(METADATA_LOCAL), 'mode': PLAN_MODE,
          'limit_per_split': 60, 'seed': 42, 'usage_rights': USAGE_RIGHTS,
          'repository_commit': subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()}
command = [PYTHON, '-m', 'deepfake_dtw.lavdf', '--metadata', str(METADATA_LOCAL),
           '--media-root', str(MEDIA), '--feature-root', str(FEATURES), '--output', str(PLAN),
           '--limit-per-split', '60', '--seed', '42']
if PLAN_MODE == 'source-group-only':
    command += ['--source-group-only', '--usage-rights', USAGE_RIGHTS]
else:
    assert PROVENANCE.is_file(), f'Identity mode needs an audited sidecar: {PROVENANCE}'
    inputs['provenance_sha256'] = digest_file(PROVENANCE)
    command += ['--provenance', str(PROVENANCE)]
if PLAN.exists():
    assert PLAN_STAMP.is_file() and json.loads(PLAN_STAMP.read_text()) == inputs, 'Existing plan is not bound to these inputs; preserve it and choose a new PLAN path.'
    print('Reusing existing plan with matching input hashes.')
else:
    assert not PLAN_STAMP.exists(), 'Plan stamp exists without plan; preserve it and choose a new PLAN path.'
    subprocess.run(command, cwd=REPO, env=PROJECT_ENV, check=True)
    with PLAN_STAMP.open('x') as stream:
        json.dump(inputs, stream, indent=2)
records = json.loads(PLAN.read_text())
print('Scope:', PLAN_MODE, '(identity disjointness is not established)' if PLAN_MODE == 'source-group-only' else '(conditional on the audited sidecar)')
print('Planned split, label and manipulation counts:', dict(Counter((r['partition'], r['label'], r['manipulation']) for r in records)))


Scope: source-group-only (identity disjointness is not established)
Planned split, label and manipulation counts: {('train', 'real', 'real'): 30, ('train', 'fake', 'both_modified'): 11, ('train', 'fake', 'video_modified'): 9, ('train', 'fake', 'audio_modified'): 10, ('validation', 'real', 'real'): 30, ('validation', 'fake', 'audio_modified'): 11, ('validation', 'fake', 'video_modified'): 10, ('validation', 'fake', 'both_modified'): 9, ('test', 'real', 'real'): 30, ('test', 'fake', 'video_modified'): 12, ('test', 'fake', 'both_modified'): 10, ('test', 'fake', 'audio_modified'): 8}


## Verify archive and extract the selected videos

The full archive hash is checked before extraction. Only the explicitly planned regular MP4 members are copied; paths and existing files are validated. Rerunning reuses existing files only after comparing them with the archive. A mismatch stops for manual inspection; nothing is overwritten. Hashing a 25.61 GB file takes time, especially on Drive.


In [29]:
from pathlib import Path
import subprocess

print("Archive exists:", Path(ARCHIVE).is_file())
print("Plan exists:", Path(PLAN).is_file())
print("Media folder exists:", Path(MEDIA).is_dir())

subprocess.run(
    [
        PYTHON,
        "-m",
        "deepfake_dtw.lavdf_archive",
        "--archive", str(ARCHIVE),
        "--plan", str(PLAN),
        "--media-root", str(MEDIA),
        "--sha256", ARCHIVE_SHA256,
    ],
    cwd=REPO,
    env=PROJECT_ENV,
    check=True
)


Archive exists: True
Plan exists: True
Media folder exists: True


CalledProcessError: Command '['/content/lavdf-cuda-env/bin/python', '-m', 'deepfake_dtw.lavdf_archive', '--archive', '/content/drive/MyDrive/deepfake-private/lavdf/archive/LAV-DF.tar', '--plan', '/content/drive/MyDrive/deepfake-private/lavdf/plan-source-groups.json', '--media-root', '/content/drive/MyDrive/deepfake-private/lavdf/media', '--sha256', '2abc0646cf1469f0bebec60a792fec8b6d670cd9e6133105139110c4316b20a4']' returned non-zero exit status 1.

In [30]:
import subprocess

cmd = [
    PYTHON,
    "-m",
    "deepfake_dtw.lavdf_archive",
    "--archive", str(ARCHIVE),
    "--plan", str(PLAN),
    "--media-root", str(MEDIA),
    "--sha256", ARCHIVE_SHA256,
]

print("Running module:", cmd[2], flush=True)

result = subprocess.run(
    cmd,
    cwd=REPO,
    env=PROJECT_ENV,
    capture_output=True,
    text=True
)

print("EXIT CODE:", result.returncode)
print("\n=== OUTPUT ===")
print(result.stdout[-6000:])
print("\n=== ERRORS ===")
print(result.stderr[-6000:])

Running module: deepfake_dtw.lavdf_archive
EXIT CODE: 1

=== OUTPUT ===
Verifying full archive SHA-256; this can take several minutes.


=== ERRORS ===
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/deepfake-detection-dtw/src/deepfake_dtw/lavdf_archive.py", line 106, in <module>
    main()
  File "/content/deepfake-detection-dtw/src/deepfake_dtw/lavdf_archive.py", line 102, in main
    print(json.dumps(extract_selected(args.archive, records, args.media_root, args.sha256), indent=2))
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/deepfake-detection-dtw/src/deepfake_dtw/lavdf_archive.py", line 26, in extract_selected
    if sha256(archive) != expected_sha256.lower():
       ^^^^^^^^^^^^^^^
  File "/content/deepfake-detection-dtw/src/deepfake_dtw/lavdf_archive.py", line 17, in sha256
    with open(path, 'rb') as stream:


In [31]:
from pathlib import Path
import subprocess
import os

archive = Path('/content/drive/MyDrive/deepfake-private/lavdf/archive/LAV-DF.tar')
folder = archive.parent

print("=== COLAB CHECK ===")
print("Drive mounted:", Path('/content/drive/MyDrive').is_dir())
print("Archive folder exists:", folder.is_dir())
print("Archive exists:", archive.is_file())

if folder.is_dir():
    print("Files in archive folder:")
    for item in folder.iterdir():
        print(" ", item.name)

if archive.is_file():
    print("Archive size GB:", round(archive.stat().st_size / 1e9, 2))

print("\n=== PROJECT PYTHON CHECK ===")

code = """
from pathlib import Path
p = Path('/content/drive/MyDrive/deepfake-private/lavdf/archive/LAV-DF.tar')
print('Exists:', p.is_file())
if p.is_file():
    print('Size GB:', round(p.stat().st_size / 1e9, 2))
"""

result = subprocess.run(
    [PYTHON, "-c", code],
    env=PROJECT_ENV,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("Exit code:", result.returncode)

=== COLAB CHECK ===
Drive mounted: True
Archive folder exists: True
Archive exists: False
Files in archive folder:
  .cache

=== PROJECT PYTHON CHECK ===
Exists: False


Exit code: 0


In [32]:
from pathlib import Path

archive_dir = Path(
    '/content/drive/MyDrive/deepfake-private/lavdf/archive'
)

cache_dir = archive_dir / '.cache' / 'huggingface'

print("=== HUGGING FACE CACHE ===")

if cache_dir.exists():
    for p in cache_dir.rglob('*'):
        if p.is_file():
            print(
                p.relative_to(archive_dir),
                "| Size:",
                round(p.stat().st_size / 1e9, 3),
                "GB"
            )
else:
    print("No Hugging Face cache directory found")

print("\n=== OTHER POSSIBLE ARCHIVE LOCATIONS ===")

for location in [
    '/content/lavdf-archive/LAV-DF.tar',
    '/content/drive/MyDrive/deepfake-private/lavdf/archive/LAV-DF.tar',
    '/content/drive/MyDrive/deepfake-private/lavdf/LAV-DF.tar',
]:
    p = Path(location)
    print(location, "FOUND" if p.is_file() else "MISSING")

=== HUGGING FACE CACHE ===
.cache/huggingface/.gitignore | Size: 0.0 GB
.cache/huggingface/download/LAV-DF.tar.lock | Size: 0.0 GB
.cache/huggingface/download/LAV-DF.tar.metadata | Size: 0.0 GB

=== OTHER POSSIBLE ARCHIVE LOCATIONS ===
/content/lavdf-archive/LAV-DF.tar MISSING
/content/drive/MyDrive/deepfake-private/lavdf/archive/LAV-DF.tar MISSING
/content/drive/MyDrive/deepfake-private/lavdf/LAV-DF.tar MISSING


In [33]:
import os
import subprocess
from pathlib import Path

local_dir = Path('/content/lavdf-archive')
local_dir.mkdir(parents=True, exist_ok=True)

download_code = """
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id='ControlNet/LAV-DF',
    repo_type='dataset',
    filename='LAV-DF.tar',
    local_dir='/content/lavdf-archive'
)
print('Downloaded:', path)
"""

env = dict(PROJECT_ENV)
env['HF_HOME'] = str(HF_HOME)

subprocess.run(
    [PYTHON, '-c', download_code],
    cwd=REPO,
    env=env,
    check=True
)

CompletedProcess(args=['/content/lavdf-cuda-env/bin/python', '-c', "\nfrom huggingface_hub import hf_hub_download\n\npath = hf_hub_download(\n    repo_id='ControlNet/LAV-DF',\n    repo_type='dataset',\n    filename='LAV-DF.tar',\n    local_dir='/content/lavdf-archive'\n)\nprint('Downloaded:', path)\n"], returncode=0)

In [34]:
from pathlib import Path

folder = Path('/content/lavdf-archive')

for file in folder.rglob('*'):
    if file.is_file() and file.stat().st_size > 1_000_000:
        print(
            file.name,
            round(file.stat().st_size / 1e9, 2),
            'GB'
        )

LAV-DF.tar 25.61 GB


In [35]:
import subprocess
from pathlib import Path

local_archive = Path('/content/lavdf-archive/LAV-DF.tar')

assert local_archive.is_file(), "Dataset archive missing"

result = subprocess.run(
    [
        PYTHON,
        "-m",
        "deepfake_dtw.lavdf_archive",
        "--archive", str(local_archive),
        "--plan", str(PLAN),
        "--media-root", str(MEDIA),
        "--sha256", ARCHIVE_SHA256,
    ],
    cwd=REPO,
    env=PROJECT_ENV,
    check=True
)

print("Archive verification and extraction completed.")

Archive verification and extraction completed.


## Extract and audit before training

The CUDA path can still be slow and memory intensive; verify the required GPU diagnostic and model smoke-test cells first. It must use the verified pretrained weights, and each failure halts the run; never relabel or silently discard failed records. Resolve extraction failures and re-audit provenance and split integrity before training. The JSON reports and features contain sensitive metadata; keep them private.

Completed feature/report pairs are revalidated against video and checkpoint hashes before reuse. If an interruption leaves only one file, preserve and inspect it before moving it aside and retrying; the notebook never silently overwrites partial output.


In [36]:
from pathlib import Path

DINO_SOURCE = Path('/content/dinov2-src')
print('DINOv2 source:', DINO_SOURCE)
print('Planned videos:', len(records))

DINOv2 source: /content/dinov2-src
Planned videos: 180


In [47]:
DATASET_VERIFIED = False
assert globals().get('GPU_DIAGNOSTIC_PASSED', False) and globals().get('MODEL_SMOKE_PASSED', False), 'Run both GPU checks first'
env = dict(PROJECT_ENV, OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2', XFORMERS_DISABLED='1', CUBLAS_WORKSPACE_CONFIG=':4096:8')
for r in records:
    out = Path(r['feature_path']); report = REPORTS / (r['sample_id'] + '.json')
    if out.exists() and report.exists():
        # Validate existing feature content, provenance, source hash and quality report before reuse.
        verify_script = '''import hashlib,json,sys
from deepfake_dtw.training import read_features
item=read_features(sys.argv[1])
with open(sys.argv[2], 'rb') as f: media_hash=hashlib.file_digest(f,'sha256').hexdigest()
with open(sys.argv[3]) as f: report=json.load(f)
assert item['input_sha256']==media_hash, 'Cached feature does not match video'
assert item['provenance']['audio']==sys.argv[4] and item['provenance']['video']==sys.argv[5], 'Cached encoder hashes differ'
assert report.get('status')=='features_extracted' and report.get('features_written') is True, 'Incomplete extraction report'
'''
        subprocess.run([PYTHON, '-c', verify_script, str(out), r['video_path'], str(report), 'd3dd57d32accea0b295c96e26691aa14d8822fac7d9d27d5dc00b4ca2826dd03', DINO_SHA256.lower()], cwd=REPO, env=env, check=True)
        print('Reused verified extraction:', r['sample_id'], flush=True)
        continue
    assert not out.exists() and not report.exists(), 'An incomplete extraction exists. Preserve it, inspect it and move it aside before retrying.'
    with report.open('x') as stream:
        subprocess.run([PYTHON, '-m', 'deepfake_dtw.extract', r['video_path'], '--audio-checkpoint', str(WHISPER), '--visual-checkpoint', str(DINO), '--visual-format', 'dinov2_vits14', '--dinov2-source', str(DINO_SOURCE), '--visual-sha256', DINO_SHA256, '--authorized-models', '--device', 'cuda', '--output', str(out)], cwd=REPO, env=env, stdout=stream, check=True)
print('Feature extraction completed for all planned clips; inspect private quality reports.')


Reused verified extraction: 307ef5ec6b3364d3b969e13ea273734878d3ef1ac6b3dd4dc75b90c35f10f941
Reused verified extraction: 38431c4ac1315dae7bd699f7e788745e55d34ca0377637c2c8262e2d76fc17e7
Reused verified extraction: 1fdda0dbea980798eba83d08ebffa581aa1fb658d212845681dc5adb98e9ae12
Reused verified extraction: df44fed1f1661f1e2a2dbc21e5957b1147f036e07e31b202cb49fddc8816f6ef
Reused verified extraction: 81ba08dd158874e288de8ec0563a8c3779d78de095b8ea1643929c41a38cb337
Reused verified extraction: 83ea1cac2a18d882e966f9b795d820860411cda81e0b273fb7b58e695f99647d
Reused verified extraction: c43cc848f8252cc45b355f716921d8f06ed2e62d4f4701777677352e363f600a
Reused verified extraction: e14c4bd659f9b9b160ea4c84eba7d3fceb1456b72e9e0d131c0a6e396d3e1768
Reused verified extraction: 8d63de1e3a2805d8d026700ece4add0d895e22377c7ffe6dde5640c9b92bf065
Reused verified extraction: eda15acdf59fa44e6125b548853f60e6a2cc4ded7edeb3cebbd09190464a02af
Reused verified extraction: 8d5efcf23e4b643bda5e17190e6fa19f0b3b1a6071

AssertionError: An incomplete extraction exists. Preserve it, inspect it and move it aside before retrying.

In [48]:
import json
from pathlib import Path

for r in records:
    feature = Path(r["feature_path"])
    report = REPORTS / f"{r['sample_id']}.json"

    if feature.exists() != report.exists():
        print("BLOCKED SAMPLE:", r["sample_id"])
        print("VIDEO:", r["video_path"])
        print("FEATURE EXISTS:", feature.exists())
        print("REPORT EXISTS:", report.exists())

        if report.is_file():
            try:
                data = json.loads(report.read_text())
                print("STATUS:", data.get("status"))
                print("REASON:", data.get("reason"))
                print("FEATURES WRITTEN:", data.get("features_written"))
            except json.JSONDecodeError:
                print("REPORT CONTENT:")
                print(report.read_text(errors="replace")[-1500:])
        break
else:
    print("No incomplete feature/report pair found")

BLOCKED SAMPLE: 1877a066e2cdbc38c27c865d013dd713b07835bcc06f58640011dc2a67d1543d
VIDEO: /content/drive/MyDrive/deepfake-private/lavdf/media/train/039298.mp4
FEATURE EXISTS: False
REPORT EXISTS: True
STATUS: face_unusable
REASON: No reliable single-face mouth track; frames were not fabricated.
FEATURES WRITTEN: False


In [46]:
import json
from pathlib import Path

complete = []
incomplete = []
pending = []

for r in records:
    feature = Path(r["feature_path"])
    report = REPORTS / f"{r['sample_id']}.json"

    if feature.is_file() and report.is_file():
        complete.append(r)
    elif feature.exists() or report.exists():
        incomplete.append((r, feature, report))
    else:
        pending.append(r)

print("Both files present:", len(complete))
print("Incomplete pairs:", len(incomplete))
print("Not started:", len(pending))

for r, feature, report in incomplete[:5]:
    print("\nSample:", r["sample_id"])
    print("Video:", r["video_path"])
    print("Feature exists:", feature.exists())
    print("Report exists:", report.exists())

    if report.is_file():
        try:
            data = json.loads(report.read_text())
            print("Status:", data.get("status"))
            print("Reason:", data.get("reason"))
            print("Features written:", data.get("features_written"))
        except json.JSONDecodeError:
            print("Incomplete or invalid report JSON")
            print(report.read_text(errors="replace")[-1500:])

Both files present: 20
Incomplete pairs: 1
Not started: 159

Sample: 1877a066e2cdbc38c27c865d013dd713b07835bcc06f58640011dc2a67d1543d
Video: /content/drive/MyDrive/deepfake-private/lavdf/media/train/039298.mp4
Feature exists: False
Report exists: True
Status: face_unusable
Reason: No reliable single-face mouth track; frames were not fabricated.
Features written: False


In [44]:
import json
from pathlib import Path

sample_id = "1877a066e2cdbc38c27c865d013dd713b07835bcc06f58640011dc2a67d1543d"

report = REPORTS / f"{sample_id}.json"

print("Report exists:", report.is_file())

if report.is_file():
    contents = report.read_text(errors="replace")

    try:
        data = json.loads(contents)
        print("Status:", data.get("status"))
        print("Reason:", data.get("reason"))
        print("Features written:", data.get("features_written"))
    except json.JSONDecodeError:
        print("Report is not valid JSON.")
        print("Last 3000 characters:")
        print(contents[-3000:])

Report exists: True
Status: face_unusable
Reason: No reliable single-face mouth track; frames were not fabricated.
Features written: False


In [40]:
from pathlib import Path

complete = []
incomplete = []
pending = []

for r in records:
    feature = Path(r["feature_path"])
    report = REPORTS / (r["sample_id"] + ".json")

    if feature.exists() and report.exists():
        complete.append(r["sample_id"])
    elif feature.exists() or report.exists():
        incomplete.append((r["sample_id"], feature, report))
    else:
        pending.append(r["sample_id"])

print("Both files present:", len(complete))
print("Incomplete pairs:", len(incomplete))
print("Not started:", len(pending))

for sample_id, feature, report in incomplete[:5]:
    print("\nSample:", sample_id)
    print("Feature exists:", feature.exists())
    print("Report exists:", report.exists())

    if report.exists():
        print("Report size:", report.stat().st_size, "bytes")
        print("Report contents:")
        print(report.read_text(errors="replace")[-2000:])

Both files present: 0
Incomplete pairs: 1
Not started: 179

Sample: 307ef5ec6b3364d3b969e13ea273734878d3ef1ac6b3dd4dc75b90c35f10f941
Feature exists: False
Report exists: True
Report size: 25560 bytes
Report contents:
       14.92,
        14.96,
        15.0,
        15.04,
        15.08,
        15.12,
        15.16,
        15.2,
        15.24,
        15.28,
        15.32,
        15.36,
        15.4,
        15.44,
        15.48,
        15.52,
        15.56,
        15.6,
        15.64,
        15.68,
        15.72,
        15.76,
        15.8,
        15.84,
        15.88,
        15.92,
        15.96,
        16.0,
        16.04,
        16.08,
        16.12,
        16.16,
        16.2,
        16.24,
        16.28,
        16.32,
        16.36,
        16.4,
        16.44,
        16.48,
        16.52,
        16.56,
        16.6,
        16.64,
        16.68,
        16.72,
        16.76,
        16.8,
        16.84,
        16.88,
        16.92,
        16.96,
        17.0,


In [41]:
import os

PROJECT_ENV["MPLBACKEND"] = "Agg"
os.environ["MPLBACKEND"] = "Agg"

print("Matplotlib backend fixed:", PROJECT_ENV["MPLBACKEND"])

Matplotlib backend fixed: Agg


In [42]:
from pathlib import Path
from datetime import datetime, timezone

sample_id = "307ef5ec6b3364d3b969e13ea273734878d3ef1ac6b3dd4dc75b90c35f10f941"

report = REPORTS / f"{sample_id}.json"
backup_dir = REPORTS / "failed-attempts"
backup_dir.mkdir(parents=True, exist_ok=True)

if report.is_file():
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    backup = backup_dir / f"{sample_id}-{timestamp}.json"

    report.rename(backup)
    print("Failed report preserved:", backup)
else:
    print("No failed report at original location")

Failed report preserved: /content/drive/MyDrive/deepfake-private/lavdf/reports/failed-attempts/307ef5ec6b3364d3b969e13ea273734878d3ef1ac6b3dd4dc75b90c35f10f941-20261108T135543Z.json


## Optional controlled timing audit

This runs measured ±0.2-second audio-timestamp perturbations on selected genuine training and validation clips only; the test partition remains untouched. It checks candidate audio-energy/lip-aperture responses, but cannot certify correspondence or support a detector ablation by itself. Review the private report and its failures. No DTW ablation or improvement claim is produced by this cell.


In [ ]:
# Exploratory timing audit; uses real selected videos, never the held-out test partition.
DTW_AUDIT = REPORTS / 'timing-audit.json'
if not DTW_AUDIT.exists():
    subprocess.run([PYTHON, '-m', 'deepfake_dtw.dtw_validation', str(PLAN), str(DTW_AUDIT), '--limit-per-partition', '20', '--seed', '42'], cwd=REPO, env=env, check=True)
else:
    saved_audit = json.loads(DTW_AUDIT.read_text())
    assert saved_audit.get('manifest_sha256') == digest_file(PLAN), 'Existing timing audit belongs to a different plan; choose a new report path.'
    by_id = {r['sample_id']: r for r in records}
    for section in saved_audit['partitions'].values():
        for sample in section['samples']:
            assert digest_file(Path(by_id[sample['sample_id']]['video_path'])) == sample['video_sha256'], 'Timing audit source video changed.'
    print('Reusing matching private timing audit; this does not establish DTW validity.')


## Train and final evaluation

Review the private quality reports, labels, manipulation counts, original links and partition audit before training. The CUDA trainer fits normalization on training data only and uses validation for checkpoint and threshold selection. Source-group-only evaluation cannot establish unseen-identity performance. The test partition must not guide model or threshold choices. Outputs remain private; sigmoid scores are not calibrated probabilities.

The trainer writes a private epoch progress checkpoint and resumes only if the manifest, feature/video hashes, configuration, PyTorch version and device match. If a final checkpoint exists without a report, preserve both before choosing new output paths.


In [ ]:
# Validate all media/features and split provenance before training; inspect the private quality reports as well.
DATASET_VERIFIED = False
subprocess.run([PYTHON, '-m', 'deepfake_dtw.dataset', 'validate', str(PLAN)], cwd=REPO, env=env, check=True)
DATASET_VERIFIED = True


In [ ]:
assert globals().get('GPU_DIAGNOSTIC_PASSED', False) and globals().get('MODEL_SMOKE_PASSED', False) and globals().get('DATASET_VERIFIED', False), 'Complete GPU checks and dataset audit first'
checkpoint = PRIVATE / 'lavdf' / 'baseline.pt'
training_report = REPORTS / 'training.json'
training_progress = PRIVATE / 'lavdf' / 'baseline.progress.pt'
if checkpoint.exists() and training_report.exists():
    saved = json.loads(training_report.read_text())
    assert saved.get('checkpoint_sha256') == digest_file(checkpoint) and saved.get('manifest_sha256') == digest_file(PLAN), 'Existing training output does not match the current checkpoint or manifest; preserve it and choose new paths.'
    print('Reusing verified training output. Review validation results before final evaluation.')
else:
    assert not checkpoint.exists() and not training_report.exists(), 'Incomplete training output exists. Preserve and inspect it before moving it aside and retrying.'
    output = subprocess.check_output([PYTHON, '-m', 'deepfake_dtw.training', 'train', str(PLAN), str(checkpoint), '--seed', '42', '--epochs', '10', '--device', 'cuda', '--progress', str(training_progress)], cwd=REPO, env=env, text=True)
    report_data = json.loads(output)
    assert report_data['checkpoint_sha256'] == digest_file(checkpoint) and report_data['manifest_sha256'] == digest_file(PLAN)
    with training_report.open('x') as stream:
        json.dump(report_data, stream, indent=2)
    print('Training report saved privately; review validation before final evaluation.')


In [ ]:
assert globals().get('DATASET_VERIFIED', False), 'Complete the dataset audit first'
# Run only after reviewing validation and freezing all model choices.
report = REPORTS / 'held-out-test.json'
assert checkpoint.is_file(), 'Complete training first'
# Verify frozen artifact bindings without parsing held-out feature arrays.
verify_frozen = """import hashlib,json,sys,torch
with open(sys.argv[1], 'rb') as f: manifest_sha=hashlib.file_digest(f,'sha256').hexdigest()
with open(sys.argv[1]) as f: rows=json.load(f)
state=torch.load(sys.argv[2],map_location='cpu',weights_only=True)
assert state.get('format')=='deepfake_dtw_classifier_v2' and state['manifest_sha256']==manifest_sha
for row in rows:
    for path_key, hash_key in (('video_path','video_hashes'),('feature_path','feature_hashes')):
        with open(row[path_key],'rb') as f: value=hashlib.file_digest(f,'sha256').hexdigest()
        assert state[hash_key][row['sample_id']]==value, 'Frozen video or feature has changed'
"""
subprocess.run([PYTHON, '-c', verify_frozen, str(PLAN), str(checkpoint)], cwd=REPO, env=env, check=True)
if report.exists():
    saved = json.loads(report.read_text())
    assert saved.get('checkpoint_sha256') == digest_file(checkpoint) and saved.get('manifest_sha256') == digest_file(PLAN) and saved.get('partition') == 'test', 'Existing test report does not match this frozen experiment; preserve it and choose a new report path.'
    print('Reusing matching held-out report; no second evaluation was run.')
else:
    with report.open('x') as stream:
        subprocess.run([PYTHON, '-m', 'deepfake_dtw.training', 'evaluate', str(PLAN), str(checkpoint), '--partition', 'test', '--device', 'cuda'], cwd=REPO, env=env, stdout=stream, check=True)
    print('Held-out report saved privately.')
